må ha rank_bm25 
https://www.geeksforgeeks.org/nlp/what-is-bm25-best-matching-25-algorithm/

In [9]:
#pip install rank_bm25

In [10]:
# importing packages 
import re
import string
from collections import Counter
from typing import List, Dict, Tuple
 
import numpy as np
import pandas as pd
from tqdm import tqdm
from rank_bm25 import BM25Okapi
from transformers import pipeline

In [11]:
#  path to the data
PASSAGE_CORPUS_PATH = "data/passage_corpus.parquet"
GOLD_MAPPING_PATH = "data/gold_mapping.parquet"
OUTPUT_PATH = "results/baseline_results.csv"

# evaluation recall of the k retrieved passages
TOP_K_VALUES = [1, 3, 5, 10, 20]

# model for question answering
QA_MODEL = "deepset/bert-base-cased-squad2"

# number of questions to evaluate on (set to None to evaluate on all questions) but we can set it to a smaller number for testing
MAX_QUESTIONS = None

# preprocessing text

https://www.geeksforgeeks.org/python/re-findall-in-python/

In [12]:
def tokenize(text: str) -> List[str]:
    # simple tokenization for BM25 

    # Convert to lowercase
    text = text.lower()
    # Remove punctuation and split
    tokens = re.findall(r'\b\w+\b', text)
    return tokens

In [13]:
def normalize_answer(text: str) -> str:
    # normalize the answer for evaluation and this aslo follow with the tokenization for BM25

    def remove_articles(text):
        return re.sub(r'\b(a|an|the)\b', ' ', text) # removing a, an, the 

    def remove_punctuation(text):
        return ''.join(ch for ch in text if ch not in string.punctuation) # removing punctuation

    def remove_whitespace(text):
        return ' '.join(text.split())  # removing extra whitespace ( if there are multiple spaces, it will be reduced to a single space)
    
    return remove_whitespace(remove_punctuation(remove_articles(text.lower()))) 


# Evaluation metric 


In [14]:
def compute_exact_match(prediction: str, ground_truth: str) -> float:
    # compute exact match score between the prediction and the ground truth answer

    return float(normalize_answer(prediction) == normalize_answer(ground_truth))

In [15]:
def compute_f1(prediction: str, ground_truth: str) -> float:
    
    # compute F1 score between the prediction and the ground truth answer
    pred_tokens = normalize_answer(prediction).split()
    gold_tokens = normalize_answer(ground_truth).split()
    
    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return float(pred_tokens == gold_tokens)
    
    common = Counter(pred_tokens) & Counter(gold_tokens) # count the number of common tokens between the prediction and the ground truth answer
    num_same = sum(common.values())
    
    if num_same == 0:
        return 0.0
    
    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)
    f1 = (2 * precision * recall) / (precision + recall)
    
    return f1


In [16]:
def compute_recall_at_k(retrieved_passages: List[int], gold_ids: List[int], k: int) -> float:

    top_k_retrieved = set(retrieved_passages[:k]) # get the top k retrieved passages
    gold_ids_set = set(gold_ids) # convert gold ids to a set

    return float(len(top_k_retrieved & gold_ids_set) > 0) # compute the recall at k by checking if there is any intersection between the top k retrieved passages and the gold passage ids, and return 1.0 if there is at least one match, otherwise return 0.0
    


    """if len(gold_ids_set) == 0:
        return 0.0

    recall = len(top_k_retrieved & gold_ids_set) / len(gold_ids_set)
    return recall # compute the recall at k by calculating the intersection of the top k retrieved passages and the gold passage ids, and dividing by the number of gold passage ids

    # could use this and just remove the return statement above if we want to compute the recall as the proportion of gold passage ids that are retrieved in the top k, but for this task we only care about whether at least one gold passage id is retrieved in the top k, so we can return 1.0 if there is at least one match and 0.0 otherwise
"""


## the retriver BM25

Very relevant use this 
https://stackoverflow.com/questions/61877065/implementation-of-okapi-bm25-in-python
https://github.com/crawlab-team/bm25/blob/main/bm25okapi.go
https://docs.langchain.com/oss/python/integrations/retrievers/bm25 # okapi for wikipedia 

Somewhat relevant: 
https://github.com/Rohith-2/bm25-fusion/blob/main/src/bm25_fusion/core.py

Not so relevant but take a look at this for furher reseach
https://github.com/xhluca/bm25s/tree/main/bm25s
https://developers.llamaindex.ai/python/framework/integrations/retrievers/bm25_retriever/   # from llama_index.retrievers.bm25 import BM25Retriever



"Okapi BM25" is the standard, variations exist to improve it:BM25F: A version that handles multiple fields (e.g., title, body, anchor text) with different importance.BM25+: Developed to fix a deficiency where long documents that match the query term are unfairly scored compared to shorter documents.rank_bm25 (Python): A popular Python library that includes implementations like BM25Okapi, BM25L, and BM25Plus

In [17]:
class BM25retriever:

    #BM25 retriever over a passage corpus 
    def __init__(self, passage_df: pd.DataFrame):

        self.passage_df = passage_df.reset_index(drop=True) 
        self.passage_idf = passage_df['passage_id'].tolist()


        self.tokenized_corpus = [tokenize(text) for text in tqdm(passage_df['text'].tolist(), desc="Tokenizing")]

        # 
        self.bm25 = BM25Okapi(self.tokenized_corpus()) # initialize the BM25 retriever with the tokenized passage corpus


    def retrieve(self, query: str, top_k: int=10) -> List[Tuple[int,float,str]]:
        tokenized_query = tokenize(query) # tokenize the query
        scores = self.bm25.get_scores(tokenized_query) # get BM25 scores for the query against all passages

        top_k_indices = np.argsort(scores)[::-1][:top_k] # get the indices of the top k passages based on the BM25 scores

        results = []
        for idx in top_k_indices:
            passage_id = self.passage_idf[idx] # get the passage id for the retrieved passage
            score = scores[idx] # get the BM25 score for the retrieved passage
            text = self.passage_df.loc[idx, 'text'] # get the text of the retrieved passage
            results.append((passage_id, score, text)) # append the passage id, score, and text of the retrieved passage to the results list
        return results



# BERT QA (baseline)

In [ ]:
class BertQA:
    def __init__(self, model_name: str = QA_MODEL):
        self.qa_pipeline = pipeline("question-answering", model=model_name, device=0) # initialize the question answering pipeline with the specified model

    def answer_question(self, question: str, context: str) -> Dict:
        try:
            result = self.qa_pipeline(question=question, context=context)
            return {
                'answer': result['answer'],
                'score': result['score'],
                'start': result['start'],
                'end': result['end']
            }
        except Exception as e:
            # Handle edge cases (empty context, etc.)
            return {
                'answer': '',
                'score': 0.0,
                'start': 0,
                'end': 0
            }

# main eval 